# Minimum graph coloring with native FlagQuantum

Keep the original five vertices, three colors, six layers, twelve angles,
100 optimizer iterations, CVaR 0.7 and 1000 optimized plus 1000 uniform samples.
Assign exactly one color to each vertex and avoid same-color neighbors.
Minimize used colors. The retained penalty is not feasibility on invalid words.


## Make the encoding visible before optimizing

The qfunc below allocates fifteen color decisions, prepares their uniform
superposition and composes six cost/mixer layers. The imported `cost` helper
in [main.py](../main.py) preserves the source's directed-edge counting and
squared conflict expression; FlagQuantum compiles that declared expression,
not an inferred graph-coloring algorithm. Low-bit-first positions group five
vertices per color. Zero compiler workspace does not mean every assignment
is feasible.

Try a different twelve-angle binding in the preview and compare color count,
occupancy violations and edge violations separately. CVaR0.7 is a low-cost-tail
objective, not a confidence level. The hundred-iteration classical optimizer
uses the shared main.py program; to change its algorithm too, make the same
edit there. A sampled feasible coloring is not an optimality certificate.


In [ ]:
from flagquantum import synth
from flagquantum.synth import (
    RX,
    Output,
    QArray,
    QBit,
    allocate,
    hadamard_transform,
    qfunc,
)
from examples.synthesis.applications.optimization.min_graph_coloring.main import cost


@qfunc
def main(params: list[float], colors: Output[QArray[QBit]]) -> None:
    allocate(15, colors)
    hadamard_transform(colors)
    for layer in range(6):
        cost(params[layer], colors)
        for bit in range(15):
            RX(params[6 + layer], colors[bit])


from examples.synthesis.applications.optimization.min_graph_coloring.main import (
    DEFAULT_PARAMS,
)
import flagquantum as fq
from examples.synthesis.applications.optimization.min_graph_coloring.overview.plots import (
    optimization_figure,
)
from examples.synthesis.applications.optimization.min_graph_coloring.overview.workflow import (
    EDGES,
    model_row,
    run_workflow,
)

print("Original graph edges:", EDGES)
program = synth.synthesize(
    main,
    classical_inputs={"params": DEFAULT_PARAMS},
    max_qubits=15,
    max_workspace=0,
    max_operations=40000,
    memory_limit_bytes=32 * 1024**2,
)
print("Actual native resources:", program.report["resources"])
print("Temporary workspace:", program.report["workspace_qubits"])


## Named native readout

The modeled fifteen-bit `colors` word uses numeric bit `5 * color + vertex`.
No physical wire numbers or manual bitstring decoding are needed for readout.


In [ ]:
initial_result = fq.run(program)
initial_readout = dict(program.probabilities(initial_result, registers=("colors",)))
print("Joint assignments:", len(initial_readout))
print("Probability mass:", sum(initial_readout.values()))


## Original-budget optimization

Every evaluation synthesizes and executes. Explicit native angles and exact
local probabilities replace private optimizer initialization and shot estimates;
trajectories need not match. This CPU run can take tens of minutes. A short CLI
smoke run is separate from this original-budget acceptance.


In [ ]:
workflow = run_workflow()
print("Initial CVaR:", workflow.trace[0].cost)
print("Final CVaR:", workflow.optimizer.value)
print("Iterations:", workflow.optimizer.iterations)
print("Actual evaluations:", workflow.optimizer.evaluations)
print("Converged:", workflow.optimizer.converged)


## Actual observations and separate reference

Do not substitute the exhaustive reference for a sampled result. Missing
feasible observations stay missing. Neither convergence nor advantage is assumed.


In [ ]:
for label, rows in (("Optimized", workflow.samples), ("Uniform", workflow.uniform)):
    print(label, "observations:", sum(row.count for row in rows))
    print(
        label, "feasible observations:", sum(row.count for row in rows if row.feasible)
    )
print("Best observed feasible:", workflow.best_feasible)
print("Separate exact reference:", model_row(workflow.exact_reference))


## Source figure roles

Use circular layout to avoid an implicit SciPy dependency. Only graph drawing
positions change. Display the actual trace, common fifty-bin sample histograms
and the best actually observed proper coloring.


In [ ]:
figure = optimization_figure(workflow)
figure
